In [14]:
from google.colab import files
uploaded = files.upload()

Saving olist_customers_dataset.csv to olist_customers_dataset (1).csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset (1).csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset (1).csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset (1).csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset (1).csv
Saving olist_orders_dataset.csv to olist_orders_dataset (1).csv
Saving olist_products_dataset.csv to olist_products_dataset (1).csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset (1).csv
Saving product_category_name_translation.csv to product_category_name_translation (2).csv


In [9]:
import pandas as pd

customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

In [10]:
for name, df in [('customers', customers), ('orders', orders),
                  ('items', items), ('payments', payments),
                  ('reviews', reviews), ('products', products),
                  ('sellers', sellers), ('category', category_translation)]:
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category: (71, 2)


In [11]:

print("=== ORDERS ===")
print(orders.info())
print(orders.isnull().sum())

print("\n=== PAYMENTS ===")
print(payments.info())
print(payments.isnull().sum())

print("\n=== ITEMS ===")
print(items.info())
print(items.isnull().sum())

=== ORDERS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivere

In [12]:

date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

orders_clean = orders.dropna(subset=['order_delivered_customer_date'])

print(f"Orders before cleaning: {len(orders)}")
print(f"Orders after cleaning: {len(orders_clean)}")

Orders before cleaning: 99441
Orders after cleaning: 96476


In [13]:

df = orders_clean.merge(customers, on='customer_id', how='left')

df = df.merge(payments, on='order_id', how='left')

df = df.merge(items, on='order_id', how='left')

print(f"Final shape: {df.shape}")
print(df.head())

Final shape: (115037, 22)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
2  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
3  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
4  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
2    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
3    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
4    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2017-10-04 19:55:

In [ ]:

payments_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_count=('payment_sequential', 'count'),
    payment_type=('payment_type', 'first')
).reset_index()

items_agg = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
).reset_index()

df = orders_clean.merge(customers, on='customer_id', how='left')
df = df.merge(payments_agg, on='order_id', how='left')
df = df.merge(items_agg, on='order_id', how='left')

print(f"Final shape: {df.shape}")

In [ ]:

df['delivery_days'] = (df['order_delivered_customer_date'] -
                       df['order_purchase_timestamp']).dt.days

df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['order_year'] = df['order_purchase_timestamp'].dt.year

df['delivery_status'] = (df['order_delivered_customer_date'] <=
                         df['order_estimated_delivery_date']).map({True: 'On Time', False: 'Late'})

print(df[['delivery_days', 'order_month', 'delivery_status']].head(10))
print(f"\nNull check:\n{df.isnull().sum()}")

In [ ]:

df = df.dropna(subset=['total_payment'])

print(f"Final clean shape: {df.shape}")
print(f"\nBasic stats:")
print(df[['total_payment', 'delivery_days', 'item_count']].describe())

In [ ]:
customer_summary = df.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment', 'sum'),
    avg_order_value=('total_payment', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    total_items=('item_count', 'sum'),
    customer_state=('customer_state', 'first'),
    customer_city=('customer_city', 'first'),
    first_order=('order_purchase_timestamp', 'min'),
    last_order=('order_purchase_timestamp', 'max'),
    on_time_rate=('delivery_status', lambda x: (x == 'On Time').mean() * 100)
).reset_index()

print(f"Customer summary shape: {customer_summary.shape}")
print(customer_summary.head())

In [ ]:

monthly_trends = df.groupby('order_month').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment', 'sum'),
    avg_order_value=('total_payment', 'mean')
).reset_index()

monthly_trends['order_month'] = monthly_trends['order_month'].astype(str)

state_summary = df.groupby('customer_state').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment', 'sum'),
    avg_delivery_days=('delivery_days', 'mean'),
    on_time_rate=('delivery_status', lambda x: (x == 'On Time').mean() * 100)
).reset_index()

print("Monthly trends shape:", monthly_trends.shape)
print(monthly_trends.head())
print("\nState summary shape:", state_summary.shape)
print(state_summary.head())

In [ ]:
customer_summary.to_csv('customer_summary.csv', index=False)
monthly_trends.to_csv('monthly_trends.csv', index=False)
state_summary.to_csv('state_summary.csv', index=False)

print("All 3 files exported!")

In [ ]:
from google.colab import files
files.download('customer_summary.csv')
files.download('monthly_trends.csv')
files.download('state_summary.csv')

# Data successfully loaded into Snowflake (OLIST_DB)
# Tables: CUSTOMER_SUMMARY, MONTHLY_TRENDS, STATE_SUMMARY
# Tableau dashboard: https://public.tableau.com/shared/2M24HQPC8?:display_count=n&:origin=viz_share_link